# LLM as Judge: Evaluating BDD Test Case Quality

This notebook uses an LLM to evaluate the three types of BDD tests generated in `test_case_generation.ipynb`:
1. **Simple** – generated from app name only  
2. **With context** – generated from app name + app context  
3. **From graph** – generated from app screen graph (nodes/edges)

We define a rubric and evaluation criteria, then call the LLM to score and compare the test sets.

In [ ]:
# Configuration: app context and API key
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
# OpenAI API key (from .env OPENAI_API_KEY or environment variable)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# App context: same as in test_case_generation.ipynb — used by the judge to assess relevance
APP_CONTEXT = """

"""

# Paths to the three generated test case files (from test_case_generation.ipynb)
APP_NAME = ""  # must match the prefix used when generating test cases
OUTPUT_DIR_SIMPLE = Path("test_cases_simple")
OUTPUT_DIR_CONTEXT = Path("test_cases_with_context")
OUTPUT_DIR_GRAPH = Path("test_cases_from_graph")

# Model used for LLM judge evaluation (e.g. gpt-4o-mini, gpt-4o)
MODEL_NAME = os.environ.get("OPENAI_MODEL")

## Rubric and evaluation criteria



Pause for a bit and think about based on what evaluation criteria and rubric, LLM should evaluate test cases

In [ ]:
# Load the three BDD test sets generated by test_case_generation.ipynb
import json

def load_test_cases(dir_path: Path, filename: str) -> list[dict]:
    path = dir_path / filename
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)

name_safe = APP_NAME.replace(" ", "_")
simple = load_test_cases(OUTPUT_DIR_SIMPLE, f"{name_safe}_test_cases.json")
with_context = load_test_cases(OUTPUT_DIR_CONTEXT, f"{name_safe}_test_cases.json")
from_graph = load_test_cases(OUTPUT_DIR_GRAPH, f"{name_safe}_graph_test_cases.json")

print(f"Simple (app name only):     {len(simple)} feature(s)")
print(f"With context:               {len(with_context)} feature(s)")
print(f"From graph:                 {len(from_graph)} feature(s)")

In [ ]:
# LLM judge: evaluation prompt and criteria
from openai import OpenAI

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    raise ValueError("Set OPENAI_API_KEY in the .env file or in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

EVALUATION_CRITERIA = """

"""

In [ ]:
# Run the LLM judge
import re

def evaluate_bdd_tests(
    app_context: str,
    simple_tests: list,
    context_tests: list,
    graph_tests: list,
    model: str = None,
) -> dict:
    """Call LLM to evaluate the three BDD test sets; returns parsed evaluation JSON."""
    if model is None:
        model = MODEL_NAME
    user_content = f"""App context (what the app is):
{app_context.strip()}

---

Test set 1 — SIMPLE (generated from app name and description only):
{json.dumps(simple_tests, indent=2)}

---

Test set 2 — WITH CONTEXT (generated from app name + app context + screenshot):
{json.dumps(context_tests, indent=2)}

---

Test set 3 — FROM GRAPH (generated from app name + app context + app screen graph + screenshot):
{json.dumps(graph_tests, indent=2)}

---

Evaluate each of the three test sets using the supplied criteria. Return only the JSON object (no markdown)."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": EVALUATION_CRITERIA},
            {"role": "user", "content": user_content},
        ],
        temperature=0.2,
    )
    text = response.choices[0].message.content.strip()
    if "```json" in text:
        text = re.search(r"```(?:json)?\s*([\s\S]*?)```", text).group(1).strip()
    elif "```" in text:
        text = re.sub(r"^```\w*\s*", "", text).strip()
        text = re.sub(r"\s*```$", "", text).strip()
    return json.loads(text)

In [ ]:
# Evaluate the three BDD test sets
result = evaluate_bdd_tests(
    app_context=APP_CONTEXT,
    simple_tests=simple,
    context_tests=with_context,
    graph_tests=from_graph,
    model=MODEL_NAME,
)

# Pretty-print evaluations
for ev in result["evaluations"]:
    print(f"\n=== {ev['test_type'].upper().replace('_', ' ')} ===")
    print(f"Overall: {ev['overall']}/5\n")
    for criterion, score in ev["scores"].items():
        justification = ev["justification"][criterion]
        print(f"  {criterion:<25} {score}  —  {justification}")


In [ ]:
# Full evaluation result (for inspection or downstream use)
result